In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import time
import logging
import sys
import json
from datetime import datetime
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# --- 1. 配置与环境设置 ---
def setup_logging(log_file='training_housing.log'):
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger(); root_logger.handlers = []; root_logger.setLevel(logging.INFO)
    file_handler = logging.FileHandler(log_file, mode='w'); file_handler.setFormatter(log_formatter); root_logger.addHandler(file_handler)
    console_handler = logging.StreamHandler(sys.stdout); console_handler.setFormatter(log_formatter); root_logger.addHandler(console_handler)

torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. 数据准备 ---
def get_housing_data_loaders(batch_size=128, test_size=0.2):
    """准备Ames Housing房价预测数据集"""
    logging.info("正在准备Ames Housing房价预测数据集...")

    # 加载数据
    train_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
    test_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

    # 合并数据集
    full_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

    # 缺失值处理（部分特征填充为"None"或0，其他数值型用中位数）
    fill_none = ["Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
                 "FireplaceQu", "GarageType", "GarageFinish", "GarageQual", "GarageCond", "PoolQC",
                 "Fence", "MiscFeature", "MasVnrType"]
    for col in fill_none:
        if col in full_df.columns:
            full_df[col] = full_df[col].fillna("None")
    fill_zero = ["BsmtFullBath", "BsmtHalfBath", "BsmtUnfSF", "GarageArea", "GarageCars", "MasVnrArea",
                 "PoolArea", "MiscVal", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch",
                 "TotRmsAbvGrd", "BedroomAbvGr", "Fireplaces", "HalfBath", "FullBath", "KitchenAbvGr",
                 "LotFrontage", "WoodDeckSF"]
    for col in fill_zero:
        if col in full_df.columns:
            full_df[col] = full_df[col].fillna(0)
    # 其他数值型特征用中位数填充
    num_cols = full_df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if full_df[col].isnull().sum() > 0:
            full_df[col] = full_df[col].fillna(full_df[col].median())

    # 删除无用特征
    cols_to_drop = ['Id', 'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'LotFrontage', 'GarageYrBlt', 'MoSold', 'YrSold']
    full_df = full_df.drop(cols_to_drop, axis=1, errors='ignore')

    # 特征工程
    full_df["TotalBath"] = full_df["BsmtFullBath"] + 0.5 * full_df["BsmtHalfBath"] + full_df["FullBath"] + 0.5 * full_df["HalfBath"]
    full_df["AllSF"] = full_df["GrLivArea"] + full_df["TotalBsmtSF"]
    full_df["AllFlrsSF"] = full_df["1stFlrSF"] + full_df["2ndFlrSF"]
    full_df["AllPorchSF"] = full_df["OpenPorchSF"] + full_df["EnclosedPorch"] + full_df["3SsnPorch"] + full_df["ScreenPorch"]

    # 类别型特征独热编码
    categorical_cols = full_df.select_dtypes(include=["object"]).columns
    full_df = pd.get_dummies(full_df, columns=categorical_cols)

    # 数值型特征对偏态进行log1p变换（绝对偏度>0.5）
    from scipy.stats import skew
    numeric_feats = full_df.select_dtypes(include=[np.number]).columns
    skewed_feats = full_df[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
    skewness = skewed_feats[abs(skewed_feats) > 0.5]
    for feat in skewness.index:
        full_df[feat] = np.log1p(full_df[feat])

    # 分离训练集和测试集
    train_df = full_df.iloc[:len(train_df), :]
    test_df = full_df.iloc[len(train_df):, :].drop(['SalePrice'], axis=1)

    # 特征和目标变量
    X = train_df.drop(['SalePrice'], axis=1)
    y = train_df['SalePrice']

    # 目标变量log1p变换
    y = np.log1p(y)

    # 标准化
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # 划分训练验证集
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=test_size, random_state=42)

    # 转换为PyTorch张量
    X_train = torch.FloatTensor(X_train)
    y_train = torch.FloatTensor(y_train.values)
    X_val = torch.FloatTensor(X_val)
    y_val = torch.FloatTensor(y_val.values)

    # 创建DataLoader
    train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
    val_dataset = torch.utils.data.TensorDataset(X_val, y_val)
    trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valloader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    logging.info("Ames Housing房价预测数据集准备完毕。")
    logging.info(f"训练样本数: {len(train_dataset)}, 验证样本数: {len(val_dataset)}")
    logging.info(f"特征维度: {X_train.shape[1]}, 目标维度: 1")

    return trainloader, valloader, X_train.shape[1]

# --- 3. 模型定义 ---
class SingleLadder(nn.Module):
    def __init__(self, depth, epsilon=0.1):
        super(SingleLadder, self).__init__()
        self.depth = depth
        self.epsilon = epsilon
        self.weights = nn.Parameter(torch.randn(depth + 1) * 0.1)

    def safe_reciprocal(self, x):
        sign = torch.sign(x)
        abs_x = torch.abs(x)
        safe_abs = torch.clamp(abs_x, min=self.epsilon)
        return sign / safe_abs

    def forward(self, x_j):
        if self.depth == 0:
            return self.weights[0] * x_j
        
        # 从最深层开始
        result = self.weights[self.depth] * x_j
        
        for k in range(self.depth - 1, 0, -1):
            denominator = self.weights[k] * x_j + self.safe_reciprocal(result)
            result = denominator
        
        # 最终层: w0 * x_j + 1/result
        final_result = self.weights[0] * x_j + self.safe_reciprocal(result)
        return final_result

class FullLadder(nn.Module):
    def __init__(self, input_dim, depth, epsilon=0.1):
        super(FullLadder, self).__init__()
        self.input_dim = input_dim
        self.depth = depth
        self.epsilon = epsilon
        self.weights = nn.Parameter(torch.randn(depth + 1, input_dim) * 0.1)
        
    def safe_reciprocal(self, x):
        sign = torch.sign(x)
        abs_x = torch.abs(x)
        safe_abs = torch.clamp(abs_x, min=self.epsilon)
        return sign / safe_abs

    def forward(self, x):
        if self.depth == 0:
            return F.linear(x, self.weights[0])

        # 从最深层开始
        result = F.linear(x, self.weights[self.depth])
        
        for k in range(self.depth - 1, 0, -1):
            a_k = F.linear(x, self.weights[k])
            denominator = a_k + self.safe_reciprocal(result)
            result = denominator
            
        # 最终层: w0^T * x + 1/result
        a_0 = F.linear(x, self.weights[0])
        final_result = a_0 + self.safe_reciprocal(result)
        return final_result

class CoFrNetDLRegression(nn.Module):
    def __init__(self, input_dim, output_dim=1, num_full_ladders=25, 
                 max_diag_ladder_depth=12, max_full_ladder_depth=12, epsilon=0.1):
        super(CoFrNetDLRegression, self).__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.num_full_ladders = num_full_ladders
        self.max_diag_ladder_depth = max_diag_ladder_depth
        self.max_full_ladder_depth = max_full_ladder_depth
        
        # 1. 对角梯子部分
        self.diag_ladders = nn.ModuleList()
        self.diag_ladder_depths = []
        
        for i in range(input_dim):
            depth = min(max_diag_ladder_depth, max(2, 5 + (i * 2) % 10))
            self.diag_ladder_depths.append(depth)
            self.diag_ladders.append(SingleLadder(depth, epsilon))
            
        # 2. 全连接梯子部分
        self.full_ladders = nn.ModuleList()
        self.full_ladder_depths = []
        
        for i in range(num_full_ladders):
            if num_full_ladders > 1:
                depth = 2 + int((i / (num_full_ladders - 1)) * (max_full_ladder_depth - 2))
            else:
                depth = 2
            depth = min(max_full_ladder_depth, max(2, depth))
            self.full_ladder_depths.append(depth)
            self.full_ladders.append(FullLadder(input_dim, depth, epsilon))
            
        # 3. 最终线性组合层 (回归任务输出维度为1)
        combined_input_dim = input_dim + num_full_ladders
        self.output_layer = nn.Linear(combined_input_dim, output_dim)
        
        nn.init.xavier_uniform_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)
        
    def forward(self, x):
        batch_size = x.shape[0]
        
        # 1. 计算对角梯子输出
        diag_outputs = []
        for j in range(self.input_dim):
            x_j = x[:, j]  
            ladder_output = self.diag_ladders[j](x_j)
            diag_outputs.append(ladder_output.unsqueeze(1))
        
        diag_features = torch.cat(diag_outputs, dim=1)  
        
        # 2. 计算全连接梯子输出
        full_outputs = []
        for ladder in self.full_ladders:
            ladder_output = ladder(x)  
            full_outputs.append(ladder_output.unsqueeze(1))
        
        full_features = torch.cat(full_outputs, dim=1)  
        
        # 3. 拼接所有特征
        combined_features = torch.cat([diag_features, full_features], dim=1)
        
        # 4. 最终线性组合 (回归任务)
        output = self.output_layer(combined_features)
        return output.squeeze()  # 回归任务输出一维

# --- 4. 训练函数 ---
def train_regression(model, trainloader, valloader, hparams, log_filepath, model_save_path):
    criterion = nn.MSELoss()  # 回归任务使用MSE损失
    optimizer = optim.Adam(model.parameters(), 
                         lr=hparams['learning_rate'], 
                         weight_decay=hparams['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=hparams['patience']//3, factor=0.5)
    
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    log_data = {
        "experiment_timestamp": datetime.now().isoformat(),
        "hyperparameters": hparams,
        "training_losses": [],
        "val_losses": [],
        "learning_rates": [],
        "best_epoch": 0
    }

    logging.info("="*30)
    logging.info("  启动 CoFrNet-DL (房价预测版) 训练流程")
    logging.info("="*30)
    
    num_epochs = hparams['num_epochs']
    loss_target = hparams['loss_target']
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.float().to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item()
        
        epoch_avg_loss = running_loss / len(trainloader)
        log_data["training_losses"].append(epoch_avg_loss)
        
        # 验证集评估
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in valloader:
                inputs, targets = inputs.to(device), targets.float().to(device)
                outputs = model(inputs)
                val_loss += criterion(outputs, targets).item()
        
        avg_val_loss = val_loss / len(valloader)
        log_data["val_losses"].append(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        log_data["learning_rates"].append(current_lr)
        
        scheduler.step(avg_val_loss)
        
        logging.info(f"Epoch {epoch+1}/{num_epochs} | 训练MSE损失: {epoch_avg_loss:.4f} | 验证MSE损失: {avg_val_loss:.4f} | 学习率: {current_lr:.2e}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict()
            log_data["best_epoch"] = epoch + 1
            patience_counter = 0
            logging.info(f"*** 新的最佳MSE损失: {best_val_loss:.4f} ***")
        else:
            patience_counter += 1
            if patience_counter >= hparams['patience']:
                logging.info(f"早停: 连续 {hparams['patience']} 轮无改进")
                break
        
        with open(log_filepath, 'w') as f:
            json.dump(log_data, f, indent=4)
        
        if best_val_loss < loss_target:
            logging.info(f"目标损失 {loss_target} 已达到。")
            break

    if best_model_state:
        logging.info(f"训练流程结束。保存最佳模型到 {model_save_path}")
        torch.save({
            'model_state_dict': best_model_state,
            'best_loss': best_val_loss,
            'hyperparameters': hparams
        }, model_save_path)
    else:
        logging.error("没有训练出任何有效模型，不进行保存。")

    return best_val_loss, best_model_state

# --- 5. 主程序 ---
if __name__ == '__main__':
    # 文件与超参数定义
    LOG_FILE = 'training_housing.log'
    JSON_LOG_FILE = 'training_housing_log.json'
    MODEL_SAVE_PATH = 'housing_cofrnet_dl.pth'
    
    HYPERPARAMETERS = {
        "num_epochs": 200,
        "loss_target": 0.01,  # MSE目标值
        "learning_rate": 5e-4,
        "weight_decay": 1e-5,
        "patience": 20,
        "num_full_ladders": 30,
        "max_diag_ladder_depth": 12,
        "max_full_ladder_depth": 12,
        "epsilon": 0.1
    }

    setup_logging(LOG_FILE)
    logging.info(f"房价预测实验超参数: {json.dumps(HYPERPARAMETERS, indent=4)}")

    trainloader, valloader, input_dim = get_housing_data_loaders()
    
    # 创建回归版的CoFrNet-DL
    cofrnet_dl = CoFrNetDLRegression(
        input_dim=input_dim, 
        output_dim=1,  # 回归任务输出维度为1
        num_full_ladders=HYPERPARAMETERS['num_full_ladders'],
        max_diag_ladder_depth=HYPERPARAMETERS['max_diag_ladder_depth'],
        max_full_ladder_depth=HYPERPARAMETERS['max_full_ladder_depth'],
        epsilon=HYPERPARAMETERS['epsilon']
    ).to(device)

    total_params = sum(p.numel() for p in cofrnet_dl.parameters())
    logging.info(f"模型总参数数量: {total_params}")

    start_time = time.time()
    best_loss, best_state_dict = train_regression(
        cofrnet_dl, trainloader, valloader,
        hparams=HYPERPARAMETERS,
        log_filepath=JSON_LOG_FILE,
        model_save_path=MODEL_SAVE_PATH
    )
    
    total_time = time.time() - start_time
    logging.info(f"总训练时间: {total_time:.2f} 秒")
    
    # 加载并验证保存的最佳模型
    if best_state_dict:
        logging.info("\n" + "="*30)
        logging.info("  开始验证已保存的最佳房价预测模型")
        logging.info("="*30)
        
        verification_model = CoFrNetDLRegression(
            input_dim=input_dim, 
            output_dim=1,
            num_full_ladders=HYPERPARAMETERS['num_full_ladders'],
            max_diag_ladder_depth=HYPERPARAMETERS['max_diag_ladder_depth'],
            max_full_ladder_depth=HYPERPARAMETERS['max_full_ladder_depth'],
            epsilon=HYPERPARAMETERS['epsilon']
        )
        
        checkpoint = torch.load(MODEL_SAVE_PATH)
        verification_model.load_state_dict(checkpoint['model_state_dict'])
        verification_model.to(device)
        verification_model.eval()

        logging.info(f"成功加载模型。保存的MSE损失: {checkpoint['best_loss']:.4f}")

        val_loss = 0.0
        criterion = nn.MSELoss()
        with torch.no_grad():
            for inputs, targets in valloader:
                inputs, targets = inputs.to(device), targets.float().to(device)
                outputs = verification_model(inputs)
                val_loss += criterion(outputs, targets).item()
        
        avg_val_loss = val_loss / len(valloader)
        logging.info(f"加载后的模型在验证集上的MSE损失: {avg_val_loss:.4f}")
        
        assert abs(avg_val_loss - checkpoint['best_loss']) < 1e-3, "验证失败！"
        logging.info("验证成功！房价预测模型可以被正确保存和复用。")
    else:
        logging.info("未能成功训练和保存房价预测模型。")

2025-08-17 14:04:19,730 [INFO ]  房价预测实验超参数: {
    "num_epochs": 200,
    "loss_target": 0.01,
    "learning_rate": 0.0005,
    "weight_decay": 1e-05,
    "patience": 20,
    "num_full_ladders": 30,
    "max_diag_ladder_depth": 12,
    "max_full_ladder_depth": 12,
    "epsilon": 0.1
}
2025-08-17 14:04:19,732 [INFO ]  正在准备Ames Housing房价预测数据集...
2025-08-17 14:04:20,005 [INFO ]  Ames Housing房价预测数据集准备完毕。
2025-08-17 14:04:20,006 [INFO ]  训练样本数: 1168, 验证样本数: 292
2025-08-17 14:04:20,007 [INFO ]  特征维度: 279, 目标维度: 1
2025-08-17 14:04:20,048 [INFO ]  模型总参数数量: 66095
2025-08-17 14:04:23,060 [INFO ]  ==============================
2025-08-17 14:04:23,061 [INFO ]    启动 CoFrNet-DL (房价预测版) 训练流程
2025-08-17 14:04:23,061 [INFO ]  ==============================
2025-08-17 14:04:28,942 [INFO ]  Epoch 1/200 | 训练MSE损失: 23.1339 | 验证MSE损失: 14.5883 | 学习率: 5.00e-04
2025-08-17 14:04:28,946 [INFO ]  *** 新的最佳MSE损失: 14.5883 ***
2025-08-17 14:04:34,640 [INFO ]  Epoch 2/200 | 训练MSE损失: 13.9909 | 验证MSE损失: 11.0542 | 学习率: 5

AssertionError: 验证失败！